# 04 — Gold dimensional model
Creates conformed dimensions and fact tables. Surrogate keys are generated deterministically using `row_number` ordered by the natural key for repeatable portfolio runs.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Customer dimension: current version only
customer_current = (spark.table("silver.customer_history")
                    .filter(F.col("is_current") == True))
w_customer = Window.orderBy("customer_id")
dim_customer = (customer_current
    .withColumn("customer_key", F.row_number().over(w_customer).cast("long"))
    .select("customer_key", "customer_id", "customer_unique_id",
            "customer_zip_code_prefix", "customer_city", "customer_state"))
dim_customer.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_customer")

# Product dimension
products = spark.table("silver.products")
w_product = Window.orderBy("product_id")
dim_product = (products
    .withColumn("product_key", F.row_number().over(w_product).cast("long"))
    .select("product_key", "product_id", "product_category_name",
            "product_name_lenght", "product_description_lenght",
            "product_photos_qty", "product_weight_g",
            "product_length_cm", "product_height_cm", "product_width_cm"))
dim_product.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_product")

# Seller dimension
sellers = spark.table("silver.sellers")
w_seller = Window.orderBy("seller_id")
dim_seller = (sellers
    .withColumn("seller_key", F.row_number().over(w_seller).cast("long"))
    .select("seller_key", "seller_id", "seller_zip_code_prefix", "seller_city", "seller_state"))
dim_seller.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_seller")

# Date dimension covering purchase dates in the source
orders = spark.table("silver.orders")
date_bounds = orders.select(
    F.to_date(F.min("order_purchase_timestamp")).alias("min_date"),
    F.to_date(F.max("order_purchase_timestamp")).alias("max_date")
).first()

dates = (spark.sql(f"SELECT explode(sequence(to_date('{date_bounds['min_date']}'), to_date('{date_bounds['max_date']}'), interval 1 day)) AS date")
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("day_of_week", F.date_format("date", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("date").isin([1, 7])))
dates.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_date")

# Fact order item: grain is one row per order_id + order_item_id
items = spark.table("silver.order_items")
fact_item = (items
    .join(orders.select("order_id", "customer_id", "order_status",
                        "order_purchase_timestamp", "order_delivered_customer_date"),
          "order_id", "left")
    .join(dim_customer.select("customer_key", "customer_id"), "customer_id", "left")
    .join(dim_product.select("product_key", "product_id"), "product_id", "left")
    .join(dim_seller.select("seller_key", "seller_id"), "seller_id", "left")
    .withColumn("purchase_date_key", F.date_format("order_purchase_timestamp", "yyyyMMdd").cast("int"))
    .select("order_id", "order_item_id", "customer_key", "product_key", "seller_key",
            "purchase_date_key", "order_status", "order_purchase_timestamp",
            "order_delivered_customer_date",
            F.col("price").alias("item_price"),
            F.col("freight_value").alias("freight_value")))
fact_item.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.fact_order_item")

# Fact payment: grain is one row per order_id + payment_sequential
fact_payment = (spark.table("silver.order_payments")
    .join(orders.select("order_id", "customer_id", "order_purchase_timestamp"), "order_id", "left")
    .join(dim_customer.select("customer_key", "customer_id"), "customer_id", "left")
    .withColumn("purchase_date_key", F.date_format("order_purchase_timestamp", "yyyyMMdd").cast("int"))
    .select("order_id", "payment_sequential", "customer_key", "purchase_date_key",
            "payment_type", "payment_installments", "payment_value"))
fact_payment.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.fact_order_payment")

print("Gold dimensional model created.")

## Gold table counts

In [ ]:
gold_tables = ["dim_customer", "dim_product", "dim_seller", "dim_date", "fact_order_item", "fact_order_payment"]
display(spark.createDataFrame([(t, spark.table(f"gold.{t}").count()) for t in gold_tables], ["table_name", "row_count"]))